In [43]:
# Dataframe and Plotting libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Machine Learning libraries
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn import model_selection
from sklearn.metrics import log_loss
from sklearn.pipeline import make_pipeline


pd.set_option('display.max_columns', None)
# from google.colab import files
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load the files into a Pandas Dataframe
train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')
ss = pd.read_csv('SampleSubmission.csv')

In [3]:
print("Train shape: "+ str(train.shape))
print("Test shape: "+ str(test.shape))

Train shape: (373, 12)
Test shape: (558, 11)


In [4]:
train.head()

,MERCHANT_CATEGORIZED_AT,MERCHANT_NAME,MERCHANT_CATEGORIZED_AS,PURCHASE_VALUE,PURCHASED_AT,IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY,USER_AGE,USER_GENDER,USER_HOUSEHOLD,USER_INCOME,USER_ID,Transaction_ID
0,2022-05-04 10:25:50.588042+00,UONSDA CHURCH MOGERE MWAYO TITHE,Rent / Mortgage,1700,2022-05-04 13:56:00+00,False,NaN,Male,3,10000,ID_ZX4DCF4K,ID_04mk78fa
1,2021-10-25 16:18:38.586837+00,PARK N GO,Transport & Fuel,100,2021-10-24 14:12:00+00,False,25.0,Female,4,90000,ID_U9WZMGJZ,ID_04xkfb07
2,2022-05-20 14:17:30.917297+00,SAFARICOM OFFERS TUNUKIWA,Data & WiFi,20,2022-05-27 12:46:00+00,False,NaN,Male,3,10000,ID_ZX4DCF4K,ID_051urwuw
3,2022-04-12 15:59:14.139347+00,ZILLIONS CREDIT LIMITED,Loan Repayment,3800,2022-02-07 12:16:00+00,False,NaN,Male,5,252000,ID_3JA0MAFB,ID_0e1om7rz
4,2022-05-04 10:24:26.709045+00,MICHAEL MUSEMBI,Bills & Fees,60,2022-05-04 15:28:00+00,False,NaN,Male,3,10000,ID_ZX4DCF4K,ID_0kfcoawb


In [5]:
test.head()

,MERCHANT_CATEGORIZED_AT,MERCHANT_NAME,PURCHASE_VALUE,PURCHASED_AT,IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY,USER_AGE,USER_GENDER,USER_HOUSEHOLD,USER_INCOME,USER_ID,Transaction_ID
0,2022-06-01 10:25:16.7131+00,KCB PAYBILL AC,150000,2022-05-05 08:29:00+00,True,NaN,Male,5,150000,ID_O8P8YS18,ID_00x9h2yx
1,2022-03-16 13:05:51.851102+00,IPAY LTD,7394,2019-10-05 16:02:00+00,False,NaN,Female,1,10000,ID_40L9OTIM,ID_01db594f
2,2022-03-10 18:25:42.112294+00,JOHN NJIRU GAKUNJU,200,2021-11-25 09:56:00+00,False,NaN,Female,1,85000,ID_1I8XYBWK,ID_07pm99no
3,2022-04-12 15:59:14.139347+00,UONSDA CHURCH OFFERING-MAIN,50,2022-04-11 14:48:00+00,False,NaN,Male,2,10000,ID_GR569FUO,ID_09jryhkd
4,2022-05-04 10:24:26.709045+00,M-SHWARI LOCK SAVINGS,10000,2022-05-04 13:59:00+00,False,NaN,Male,3,10000,ID_ZX4DCF4K,ID_0ccv924c


# Prepare

In [6]:
test.isna().sum()

MERCHANT_CATEGORIZED_AT                    0
MERCHANT_NAME                              0
PURCHASE_VALUE                             0
PURCHASED_AT                               0
IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY      0
USER_AGE                                 473
USER_GENDER                                5
USER_HOUSEHOLD                             0
USER_INCOME                                0
USER_ID                                    0
Transaction_ID                             0
dtype: int64

In [7]:
features = train[['MERCHANT_CATEGORIZED_AT',"PURCHASE_VALUE","PURCHASED_AT","IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY", 'USER_INCOME',"USER_GENDER","USER_HOUSEHOLD"]]

In [8]:
target = train["MERCHANT_CATEGORIZED_AS"]

In [9]:
features["MERCHANT_CATEGORIZED_AT"] = pd.to_datetime(features["MERCHANT_CATEGORIZED_AT"])
features["MERCHANT_CATEGORIZED_AT"] = features['MERCHANT_CATEGORIZED_AT'].dt.hour

In [10]:
features["PURCHASED_AT"] = pd.to_datetime(features["PURCHASED_AT"])
features["PURCHASED_AT"] = features['PURCHASED_AT'].dt.hour

In [11]:
def binate(x):
    if x == True:
        return 1
    elif x == False:
        return 0
    elif x == "Male":
        return 1
    elif x == "Female":
        return 0
    else:
        return 1

In [12]:
features["IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY"] = features["IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY"].apply(binate)

In [13]:
features["USER_GENDER"] = features["USER_GENDER"].apply(binate)

In [42]:
pd.get_dummies(pd.concat(test['MERCHANT_NAME'],).head().shape

(5, 314)

In [14]:
features.head()

,MERCHANT_CATEGORIZED_AT,PURCHASE_VALUE,PURCHASED_AT,IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY,USER_INCOME,USER_GENDER,USER_HOUSEHOLD
0,10,1700,13,0,10000,1,3
1,16,100,14,0,90000,0,4
2,14,20,12,0,10000,1,3
3,15,3800,12,0,252000,1,5
4,10,60,15,0,10000,1,3


In [15]:
features.isna().sum()

MERCHANT_CATEGORIZED_AT                  0
PURCHASE_VALUE                           0
PURCHASED_AT                             0
IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY    0
USER_INCOME                              0
USER_GENDER                              0
USER_HOUSEHOLD                           0
dtype: int64

# Prepare TEST

In [17]:
tfeatures = test[['MERCHANT_CATEGORIZED_AT',"PURCHASE_VALUE","PURCHASED_AT","IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY", 'USER_INCOME',"USER_GENDER","USER_HOUSEHOLD"]]

In [18]:
tfeatures["MERCHANT_CATEGORIZED_AT"] = pd.to_datetime(tfeatures["MERCHANT_CATEGORIZED_AT"])
tfeatures["MERCHANT_CATEGORIZED_AT"] = tfeatures['MERCHANT_CATEGORIZED_AT'].dt.hour

In [19]:
tfeatures["PURCHASED_AT"] = pd.to_datetime(tfeatures["PURCHASED_AT"])
tfeatures["PURCHASED_AT"] = tfeatures['PURCHASED_AT'].dt.hour

In [20]:
tfeatures["IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY"] = tfeatures["IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY"].apply(binate)

In [21]:
tfeatures["USER_GENDER"] = tfeatures["USER_GENDER"].apply(binate)

In [22]:
tfeatures.head()

,MERCHANT_CATEGORIZED_AT,PURCHASE_VALUE,PURCHASED_AT,IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY,USER_INCOME,USER_GENDER,USER_HOUSEHOLD
0,10,150000,8,1,150000,1,5
1,13,7394,16,0,10000,0,1
2,18,200,9,0,85000,0,1
3,15,50,14,0,10000,1,2
4,10,10000,13,0,10000,1,3


In [23]:
tfeatures.isna().sum()

MERCHANT_CATEGORIZED_AT                  0
PURCHASE_VALUE                           0
PURCHASED_AT                             0
IS_PURCHASE_PAID_VIA_MPESA_SEND_MONEY    0
USER_INCOME                              0
USER_GENDER                              0
USER_HOUSEHOLD                           0
dtype: int64

In [24]:
tfeatures.shape

(558, 7)

# MODEL

In [60]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import BaggingClassifier, StackingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

In [58]:
# ESTIMATORS
svm = LinearSVC()
svm = CalibratedClassifierCV(svm)

tree = DecisionTreeClassifier()

forest = RandomForestClassifier(criterion='log_loss',n_estimators=100, random_state=42)

light  = LGBMClassifier(learning_rate=0.05)

In [28]:
# ENSEMBLE
bag = BaggingClassifier(base_estimator=forest, n_estimators=10, random_state=0)


decks = [('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
         ('bgc', BaggingClassifier(base_estimator=svm, n_estimators=10, random_state=0))
        ]
stack = StackingClassifier(estimators=decks, final_estimator=LogisticRegression(), stack_method= 'predict_proba')

In [61]:
parameters = {'criterion':['gini', 'entropy', 'log_loss'], 'n_estimators':[100,500,300]}


model = XGBClassifier()
model.fit(features, target)

predictions = model.predict_proba(tfeatures)

[23:40:13] WARNING: ..\src\learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'multi:softprob' was changed from 'merror' to 'mlogloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


In [62]:
predictions.shape

(558, 13)

In [63]:
predictions[0:3]

array([[1.0424881e-01, 1.3757556e-02, 1.9734402e-01, 2.9483505e-02,
        3.8565531e-01, 2.2611898e-04, 8.6360821e-04, 1.9598897e-03,
        6.3061207e-03, 9.9825521e-04, 6.2283117e-02, 1.8969227e-01,
        7.1814293e-03],
       [4.1494153e-02, 9.9066726e-04, 7.2825479e-04, 6.9663008e-03,
        2.9442480e-04, 9.4335121e-01, 8.9258532e-04, 2.5889993e-04,
        3.4283104e-04, 1.0600148e-03, 5.5170921e-04, 1.5484507e-03,
        1.5204949e-03],
       [1.0279414e-03, 4.9612566e-04, 1.7484670e-03, 3.5182558e-04,
        1.5437703e-03, 6.7879862e-01, 2.0368946e-01, 2.0736144e-03,
        2.9926182e-04, 7.4319452e-02, 3.7267932e-04, 2.3348033e-02,
        1.1930773e-02]], dtype=float32)

In [64]:
model.classes_

array(['Bills & Fees', 'Data & WiFi', 'Education', 'Emergency fund',
       'Family & Friends', 'Going out', 'Groceries', 'Health',
       'Loan Repayment', 'Miscellaneous', 'Rent / Mortgage', 'Shopping',
       'Transport & Fuel'], dtype=object)

In [65]:
predictions_df = pd.DataFrame(predictions)
predictions_df.columns = model.classes_
predictions_df.head()


,Bills & Fees,Data & WiFi,Education,Emergency fund,Family & Friends,Going out,Groceries,Health,Loan Repayment,Miscellaneous,Rent / Mortgage,Shopping,Transport & Fuel
0,0.104249,0.013758,0.197344,0.029484,0.385655,0.000226,0.000864,0.001960,0.006306,0.000998,0.062283,0.189692,0.007181
1,0.041494,0.000991,0.000728,0.006966,0.000294,0.943351,0.000893,0.000259,0.000343,0.001060,0.000552,0.001548,0.001520
2,0.001028,0.000496,0.001748,0.000352,0.001544,0.678799,0.203689,0.002074,0.000299,0.074319,0.000373,0.023348,0.011931
3,0.100056,0.050985,0.000984,0.001143,0.006099,0.002459,0.000995,0.001987,0.001344,0.805667,0.001918,0.020313,0.006051
4,0.860578,0.000524,0.001197,0.002129,0.000390,0.004010,0.001700,0.000609,0.018589,0.021900,0.085555,0.000978,0.001842


In [34]:
def pimp(x):
#     return 1
    if x < 0.025:
        return 0.025
    elif x > .975:
        return .975
    else:
        return x

for column in predictions_df.columns:
    predictions_df[column] = predictions_df[column].apply(pimp)
predictions_df

,Bills & Fees,Data & WiFi,Education,Emergency fund,Family & Friends,Going out,Groceries,Health,Loan Repayment,Miscellaneous,Rent / Mortgage,Shopping,Transport & Fuel
0,0.157,0.132,0.035,0.079,0.420,0.025000,0.025000,0.025,0.025,0.025000,0.032,0.094,0.025000
1,0.090,0.048,0.025,0.037,0.025,0.734667,0.025000,0.025,0.025,0.025000,0.025,0.025,0.025000
2,0.037,0.025,0.025,0.025,0.025,0.424000,0.197187,0.038,0.025,0.156167,0.025,0.042,0.078647
3,0.218,0.065,0.025,0.025,0.025,0.182000,0.025000,0.025,0.025,0.354000,0.025,0.124,0.025000
4,0.307,0.075,0.025,0.056,0.025,0.031000,0.059000,0.025,0.025,0.029000,0.324,0.050,0.047000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
553,0.025,0.122,0.025,0.029,0.025,0.263000,0.263000,0.025,0.025,0.025000,0.032,0.025,0.235000
554,0.189,0.025,0.025,0.025,0.681,0.059000,0.025000,0.025,0.025,0.027000,0.025,0.025,0.027000
555,0.088,0.124,0.025,0.037,0.025,0.172000,0.025000,0.025,0.025,0.066000,0.025,0.025,0.484000
556,0.345,0.025,0.025,0.101,0.227,0.025000,0.090000,0.025,0.025,0.162000,0.025,0.025,0.025000


In [49]:
ss.head()

,Transaction_ID,Bills & Fees,Data & WiFi,Education,Emergency fund,Family & Friends,Going out,Groceries,Health,Loan Repayment,Miscellaneous,Rent / Mortgage,Shopping,Transport & Fuel
0,ID_00x9h2yx,0,0,0,0,0,0,0,0,0,0,0,0,0
1,ID_01db594f,0,0,0,0,0,0,0,0,0,0,0,0,0
2,ID_07pm99no,0,0,0,0,0,0,0,0,0,0,0,0,0
3,ID_09jryhkd,0,0,0,0,0,0,0,0,0,0,0,0,0
4,ID_0ccv924c,0,0,0,0,0,0,0,0,0,0,0,0,0


In [66]:
predictions_df['Transaction_ID'] = ss['Transaction_ID']
predictions_df.head()

,Bills & Fees,Data & WiFi,Education,Emergency fund,Family & Friends,Going out,Groceries,Health,Loan Repayment,Miscellaneous,Rent / Mortgage,Shopping,Transport & Fuel,Transaction_ID
0,0.104249,0.013758,0.197344,0.029484,0.385655,0.000226,0.000864,0.001960,0.006306,0.000998,0.062283,0.189692,0.007181,ID_00x9h2yx
1,0.041494,0.000991,0.000728,0.006966,0.000294,0.943351,0.000893,0.000259,0.000343,0.001060,0.000552,0.001548,0.001520,ID_01db594f
2,0.001028,0.000496,0.001748,0.000352,0.001544,0.678799,0.203689,0.002074,0.000299,0.074319,0.000373,0.023348,0.011931,ID_07pm99no
3,0.100056,0.050985,0.000984,0.001143,0.006099,0.002459,0.000995,0.001987,0.001344,0.805667,0.001918,0.020313,0.006051,ID_09jryhkd
4,0.860578,0.000524,0.001197,0.002129,0.000390,0.004010,0.001700,0.000609,0.018589,0.021900,0.085555,0.000978,0.001842,ID_0ccv924c


In [67]:
predictions_df = predictions_df[['Transaction_ID', 'Bills & Fees', 'Data & WiFi', 'Education', 'Emergency fund',
       'Family & Friends', 'Going out', 'Groceries', 'Health',
       'Loan Repayment', 'Miscellaneous', 'Rent / Mortgage', 'Shopping',
       'Transport & Fuel']]

In [68]:
# Create the submission csv file
predictions_df.to_csv('xgb.csv', index = False)

In [53]:
predictions_df.shape

(558, 14)

In [242]:
model.score(features,target)

0.9463806970509383

In [143]:
log_loss(ss.drop('Transaction_ID',axis=1), predictions_df.drop('Transaction_ID',axis=1))

0.0